# 3.6 Demo - User-Defined Functions
**Lesson:** Serialization

Spark runs on the JVM. A **Python UDF** forces every row to be **serialized** to a Python worker process and back. That costs CPU, it blocks whole-stage code generation, and the optimizer can't see inside the function, so there is no predicate pushdown and no constant folding.

Options, fastest first:
| Option | Runs in | Data transfer |
|---|---|---|
| Built-in functions (`pyspark.sql.functions`, SQL) | JVM, code-generated | none |
| **SQL UDF** (`CREATE FUNCTION ... RETURN <sql expr>`) | JVM, inlined into the plan | none |
| **Pandas UDF** (`@pandas_udf`) | Python, vectorized | Arrow batches |
| Arrow-optimized Python UDF (`useArrow=True`) | Python, row-by-row | Arrow batches |
| Classic Python UDF (`@udf`) | Python, row-by-row | pickled rows |

In [ ]:
from adelab.classroom import setup
from adelab import perf
from pyspark.sql import functions as F
import pandas as pd

DA = setup("3.6", reset=True)
(spark.range(3_000_000)
      .withColumn("email", F.concat(F.lit("user"), F.col("id").cast("string"), F.lit("@Example.COM ")))
      .withColumn("amount_cents", (F.rand(seed=1) * 100_000).cast("long"))
      .write.saveAsTable("payments"))
# Materialize the input once, so every variant measures only the function cost (not the scan).
# It also works around a PySpark 4.2 issue on Windows: row-at-a-time Python UDFs fed directly by a
# parallel vectorized Parquet/Delta scan can stall until the worker times out (see README).
payments = spark.table("payments").localCheckpoint()

## A. The task: normalize an email and convert cents to an amount with VAT
The same logic, five ways:

In [ ]:
from pyspark.sql.functions import udf, pandas_udf

# 1. Classic Python UDF
@udf("string")
def normalize_email_udf(e):
    return e.strip().lower() if e is not None else None

@udf("double")
def gross_udf(cents):
    return round(cents / 100 * 1.19, 2) if cents is not None else None

# 2. Arrow-optimized Python UDF (same code, Arrow serialization)
normalize_email_arrow = udf(lambda e: e.strip().lower() if e is not None else None, "string", useArrow=True)
gross_arrow = udf(lambda c: round(c / 100 * 1.19, 2) if c is not None else None, "double", useArrow=True)

# 3. Pandas UDF (vectorized: one call per Arrow batch of ~10k rows)
@pandas_udf("string")
def normalize_email_pandas(s: pd.Series) -> pd.Series:
    return s.str.strip().str.lower()

@pandas_udf("double")
def gross_pandas(c: pd.Series) -> pd.Series:
    return (c / 100 * 1.19).round(2)

# 4. SQL UDF (inlined by the optimizer)
spark.sql("CREATE OR REPLACE TEMPORARY FUNCTION normalize_email_sql(e STRING) RETURNS STRING RETURN lower(trim(e))")
spark.sql("CREATE OR REPLACE TEMPORARY FUNCTION gross_sql(c BIGINT) RETURNS DOUBLE RETURN round(c / 100 * 1.19, 2)")

variants = {
    "1 python udf":        payments.select(normalize_email_udf("email").alias("e"), gross_udf("amount_cents").alias("g")),
    "2 arrow python udf":  payments.select(normalize_email_arrow("email").alias("e"), gross_arrow("amount_cents").alias("g")),
    "3 pandas udf":        payments.select(normalize_email_pandas("email").alias("e"), gross_pandas("amount_cents").alias("g")),
    "4 sql udf":           payments.selectExpr("normalize_email_sql(email) AS e", "gross_sql(amount_cents) AS g"),
    "5 built-in":          payments.select(F.lower(F.trim("email")).alias("e"),
                                           F.round(F.col("amount_cents") / 100 * 1.19, 2).alias("g")),
}

## B. Compare
Each variant computes the same 3 M rows. Run the cell twice. The first run of each Python variant also pays to start the Python worker.

In [ ]:
results = [perf.run(df, name) for name, df in variants.items()]
display(perf.compare(*results)[["query", "seconds"]])
perf.plot(results, "seconds", "Time per UDF variant", highlight="5 built-in", note="stays in the JVM");

Look at the step between the Python variants (1-3) and the JVM ones (4-5): the gap is the cost of moving rows to Python and back.

## C. Why: look at the physical plans
- Python UDF: `BatchEvalPython`. The stage is split around it and rows are pickled
- Arrow and pandas UDF: `ArrowEvalPython`. Columnar batches
- SQL UDF and built-ins: just `Project` inside `WholeStageCodegen`. The UDF is gone, inlined

![Classic Python UDF pickles every row to a Python worker; pandas UDF sends Arrow batches; built-in and SQL UDF stay inside the JVM](../images/3.6-udf-serialization.svg)

**How to read it:** each lane is one variant from section A. The red box is the per-row pickling that makes variant 1 slow; the
blue Arrow box is why 2 and 3 are cheaper; lane 3 never leaves the JVM. The operator names on the right are what
to look for in the plans printed below.

In [ ]:
for name in ("1 python udf", "3 pandas udf", "4 sql udf"):
    print(f"===== {name}")
    variants[name].explain()

## D. UDFs block optimizations
A filter on a UDF result can't be pushed into the file scan, so data skipping is lost. The optimizer also can't reorder or simplify it.

In [ ]:
spark.sql("SELECT * FROM payments WHERE lower(trim(email)) = 'user42@example.com'").explain()
spark.table("payments").filter(normalize_email_udf("email") == "user42@example.com").explain()

## E. Guidelines
- Search `pyspark.sql.functions` first: string, date, array, map, JSON, regex, higher-order functions (`transform`, `filter`, `aggregate`) cover most needs.
- Need reusable logic? Write a **SQL UDF**. It is governed in Unity Catalog and as fast as built-ins.
- Need Python libraries (ML models, complex parsing)? Use a **pandas UDF**, or `mapInPandas` / `applyInPandas` for whole groups.
- Classic row-at-a-time Python UDFs are the last resort. If you must use one, at least set `useArrow=True`.

In [ ]:
DA.cleanup()